# OpenCV + YOLO: Custom Object Detector Training Pipeline
### Complete Kaggle GPU Training Template (Supports PASCAL VOC & Custom Datasets)

This notebook implements **Phases 6, 7 & 8** of the *OpenCV Object Detection Complete Build Roadmap*.

**Workflow:**
1. Verify GPU accelerator
2. Install/Upgrade Ultralytics
3. Set up dataset (`VOC.yaml` for 20 PASCAL VOC classes or custom `data.yaml`)
4. Execute rapid 3-epoch **Smoke Test** to verify paths and data integrity
5. Run full **Fine-Tuning** transfer learning
6. Validate model performance ($mAP_{50}$, $mAP_{50-95}$, Precision, Recall)
7. Inspect visual predictions on unseen test data
8. Download `best.pt` for local OpenCV deployment

### Step 1: Environment Setup & GPU Verification

In [ ]:
!nvidia-smi
!pip install -q -U ultralytics opencv-python

import torch
import ultralytics
from ultralytics import YOLO

print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name       : {torch.cuda.get_device_name(0)}")
print(f"Ultralytics Check : {ultralytics.__version__}")

### Step 2: Dataset Configuration
We can use `VOC.yaml` (which automatically downloads the 20-class PASCAL VOC benchmark dataset on Kaggle), or point to an uploaded custom dataset.

In [ ]:
import glob
import os

# Set DATA_YAML = 'VOC.yaml' for PASCAL VOC (20 classes: person, car, dog, etc.)
# Or use an uploaded dataset YAML
DATA_YAML = "VOC.yaml"

yaml_files = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
if yaml_files:
    DATA_YAML = yaml_files[0]
    print(f"Using uploaded dataset: {DATA_YAML}")
else:
    print(f"Using dataset: {DATA_YAML} (PASCAL VOC benchmark)")

if os.path.exists(DATA_YAML):
    with open(DATA_YAML, 'r') as f:
        print("\n--- Dataset Configuration ---")
        print(f.read())

### Step 3: Smoke Test Run (3 Epochs)
> **Roadmap Rule:** Always run a 2–5 epoch smoke test before launching a long training job. This validates dataset downloading, label formatting, batch memory, and GPU pipeline.

In [ ]:
# Load small pretrained YOLO checkpoint for transfer learning
model_smoke = YOLO("yolo11n.pt")

model_smoke.train(
    data=DATA_YAML,
    epochs=3,
    imgsz=640,
    batch=16,
    project="/kaggle/working/smoke_test",
    name="smoke_run",
    verbose=True
)
print("Smoke test completed successfully! Proceeding to baseline fine-tuning.")

### Step 4: Baseline Fine-Tuning Training Run (50 Epochs)

In [ ]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    save=True,
    project="/kaggle/working/runs",
    name="yolo_voc_train",
    plots=True,
    verbose=True
)

### Step 5: Model Validation & Metrics Inspection
Evaluate precision, recall, $mAP_{50}$ and $mAP_{50-95}$ across all 20 classes.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Run validation
metrics = model.val()
print(f"mAP@50     : {metrics.box.map50:.4f}")
print(f"mAP@50-95  : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

# Display Confusion Matrix
cm_path = "/kaggle/working/runs/yolo_voc_train/confusion_matrix.png"
if os.path.exists(cm_path):
    plt.figure(figsize=(12, 10))
    plt.imshow(Image.open(cm_path))
    plt.axis('off')
    plt.title("Confusion Matrix")
    plt.show()

### Step 6: Test Predictions on Unseen Images

In [ ]:
# Run predictions on val images
predict_results = model.predict(
    source="/kaggle/working/runs/yolo_voc_train/val_batch0_labels.jpg",
    conf=0.35,
    save=True,
    project="/kaggle/working/predictions"
)
print("Inference test complete.")

### Step 7: Export & Prepare `best.pt` for Download
Download the resulting `best.pt` to your local project folder under `models/best.pt`.

In [ ]:
import shutil

best_weights = "/kaggle/working/runs/yolo_voc_train/weights/best.pt"
target_download = "/kaggle/working/best.pt"

if os.path.exists(best_weights):
    shutil.copy2(best_weights, target_download)
    print(f"Successfully exported model to: {target_download}")
    print("Download 'best.pt' and place it in your local 'models/' directory.")
else:
    print("best.pt not found yet. Check the training run outputs.")